In [7]:
pip install -U openai-whisper torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=e962c1b7f0bc7c446b4a7ca8553030fc3307049012b6f6488a406b4c036c0a25
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [8]:
import whisper
from pathlib import Path
from typing import List, Optional

# ==========================
# CONFIGURATION (edit these)
# ==========================

AUDIO_FILE = "Lea_Seydoux_reveals_her_ultimate_icons_from_Lady_Diana_to_Daniel.mp3"
MODEL_SIZE = "small"

# If your audio is primarily French, keep LANGUAGE="fr"; else change appropriately.
LANGUAGE = "fr"

# Provide speaker names (optional). If INCLUDE_SPEAKER_NAMES is True, the script
# will assign speakers by alternating through this list (naive approach).
SPEAKERS: List[str] = [
    "Interviewer",
    "Lea Seydoux"
]

# Toggle features:
INCLUDE_SPEAKER_NAMES = False         # True -> include "Speaker: ..." in each segment line
TRANSLATE_TO_ENGLISH = True         # True -> produce English translation(s)
KEEP_SOURCE_WHEN_TRANSLATING = True # Only used if TRANSLATE_TO_ENGLISH == True.
                                     # If True, output both original-language text and English translation.
                                     # If False, output only English (single pass with task="translate").

# Output file:
OUTPUT_FILE = "transcript.txt"

# ==========================
# HELPERS
# ==========================

def format_time(seconds: float) -> str:
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins:02d}:{secs:02d}"

def assign_speaker_for_index(i: int, speakers: List[str]) -> str:
    if not speakers:
        return ""
    return speakers[i % len(speakers)]

def align_translation_to_source(source_segments, trans_segments, tolerance_sec: float = 1.0):
    """
    Create a list of pairs (source_seg, matched_translation_text).
    We match by nearest start time within tolerance. If no close match, translation text becomes "".
    """
    matched_trans_texts = []
    trans_by_start = [(t["start"], t) for t in trans_segments]
    trans_starts = [t[0] for t in trans_by_start]

    for s in source_segments:
        s_start = s["start"]
        # find nearest translation segment by start time
        if not trans_starts:
            matched_trans_texts.append("")
            continue
        # find index of minimum absolute difference
        diffs = [abs(ts - s_start) for ts in trans_starts]
        min_idx = int(diffs.index(min(diffs)))
        if diffs[min_idx] <= tolerance_sec:
            matched_trans_texts.append(trans_by_start[min_idx][1]["text"].strip())
        else:
            matched_trans_texts.append("")  # no close match
    return matched_trans_texts

# ==========================
# LOAD MODEL
# ==========================

print("Loading Whisper model...")
model = whisper.load_model(MODEL_SIZE)

# ==========================
# TRANSCRIBE / TRANSLATE
# ==========================

# Decide tasks
if TRANSLATE_TO_ENGLISH and KEEP_SOURCE_WHEN_TRANSLATING:
    # two passes: first original-language transcription, then translation
    print("Transcribing (original language)...")
    original_result = model.transcribe(
        AUDIO_FILE,
        language=LANGUAGE,
        task="transcribe",
        verbose=False
    )
    print("Transcribing (translation to English)...")
    translation_result = model.transcribe(
        AUDIO_FILE,
        task="translate",  # target will be English
        verbose=False
    )

    source_segments = original_result["segments"]
    trans_segments = translation_result["segments"]

    # align translations to source segments
    translations_aligned = align_translation_to_source(source_segments, trans_segments, tolerance_sec=1.0)

    # Build lines from aligned pairs
    script_lines = []
    for i, seg in enumerate(source_segments):
        start_time = format_time(seg["start"])
        end_time = format_time(seg["end"])
        src_text = seg["text"].strip()
        trans_text = translations_aligned[i] or ""
        if INCLUDE_SPEAKER_NAMES and SPEAKERS:
            speaker = assign_speaker_for_index(i, SPEAKERS)
            speaker_prefix = f"{speaker}: "
        else:
            speaker_prefix = ""

        if trans_text:
            # both source and translation
            line = f"[{start_time} – {end_time}] {speaker_prefix}{src_text}\n[{start_time} – {end_time}] EN: {trans_text}"
        else:
            line = f"[{start_time} – {end_time}] {speaker_prefix}{src_text}"
        script_lines.append(line)

elif TRANSLATE_TO_ENGLISH:
    # single pass translate -> text will be English
    print("Transcribing + translating to English (single pass)...")
    result = model.transcribe(
        AUDIO_FILE,
        task="translate",
        verbose=False
    )
    segments = result["segments"]

    script_lines = []
    for i, segment in enumerate(segments):
        start_time = format_time(segment["start"])
        end_time = format_time(segment["end"])
        text = segment["text"].strip()
        if INCLUDE_SPEAKER_NAMES and SPEAKERS:
            speaker = assign_speaker_for_index(i, SPEAKERS)
            line = f"[{start_time} – {end_time}] {speaker}: {text}"
        else:
            line = f"[{start_time} – {end_time}] {text}"
        script_lines.append(line)

else:
    # regular transcription in source language
    print("Transcribing audio (source language)...")
    result = model.transcribe(
        AUDIO_FILE,
        language=LANGUAGE,
        task="transcribe",
        verbose=False
    )
    segments = result["segments"]

    script_lines = []
    for i, segment in enumerate(segments):
        start_time = format_time(segment["start"])
        end_time = format_time(segment["end"])
        text = segment["text"].strip()
        if INCLUDE_SPEAKER_NAMES and SPEAKERS:
            speaker = assign_speaker_for_index(i, SPEAKERS)
            line = f"[{start_time} – {end_time}] {speaker}: {text}"
        else:
            line = f"[{start_time} – {end_time}] {text}"
        script_lines.append(line)

full_script = "\n\n".join(script_lines)

# ==========================
# SAVE TO FILE
# ==========================

Path(OUTPUT_FILE).write_text(full_script, encoding="utf-8")
print("✅ Transcription complete!")
print(f"Saved to: {OUTPUT_FILE}")

Loading Whisper model...


100%|████████████████████████████████████████| 461M/461M [00:04<00:00, 115MiB/s]


Transcribing (original language)...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 25646/25646 [03:10<00:00, 134.97frames/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribing (translation to English)...
Detected language: French


100%|██████████| 25646/25646 [02:43<00:00, 156.76frames/s]

✅ Transcription complete!
Saved to: transcript.txt
